# Experiment 5 — XGBoost Threshold Optimization

Default threshold = 0.5 is wrong for imbalanced data.
Scan thresholds 0.1–0.9 on validation set, pick best F1, apply to test set.
Cheapest technique — no retraining needed.

**Prerequisite:** Run `00_dataset_prep.ipynb` first.

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

PARAMS     = dict(n_estimators=500, max_depth=6, learning_rate=0.1,
                  random_state=42, eval_metric='auc', verbosity=0,
                  use_label_encoder=False)
THRESHOLDS = np.arange(0.1, 0.91, 0.01)
print("Experiment 5 — XGBoost Threshold Optimization")

Experiment 5 — XGBoost Threshold Optimization


In [2]:
all_results = []

for v in ['A', 'B', 'C']:
    print(f"\n[Exp5-XGB] Training Version {v}...")
    try:
        train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
        test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    except FileNotFoundError:
        print(f"ERROR: Run 00_dataset_prep.ipynb first."); raise

    X_tr_full = train.drop('label',axis=1).values; y_tr_full = train['label'].values
    X_test    = test.drop('label',axis=1).values;  y_test    = test['label'].values

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr_full, y_tr_full, test_size=0.2, random_state=42, stratify=y_tr_full)

    model = XGBClassifier(**PARAMS)
    t0    = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0

    val_probs = model.predict_proba(X_val)[:,1]
    f1_scores = [f1_score(y_val, (val_probs>=t).astype(int), pos_label=1, zero_division=0)
                 for t in THRESHOLDS]
    best_threshold = THRESHOLDS[int(np.argmax(f1_scores))]
    print(f"[Exp5-XGB] Version {v}: best_threshold={best_threshold:.2f} | val_F1={max(f1_scores):.4f}")

    probs = model.predict_proba(X_test)[:,1]
    preds = (probs >= best_threshold).astype(int)

    metrics = compute_all_metrics(y_test, probs, preds, train_time)
    metrics['Version']        = v
    metrics['Best_Threshold'] = round(best_threshold, 2)
    all_results.append(metrics)

    np.save(os.path.join(RESULTS_DIR, f'probs_exp5_xgb_version_{v}.npy'), probs)
    print_metrics(metrics, label=f"Exp5-XGB Version {v}")

print("\n[Exp5-XGB] Complete.")


[Exp5-XGB] Training Version A...
[Exp5-XGB] Version A: best_threshold=0.36 | val_F1=0.7704
[Exp5-XGB Version A] AUC=0.8215 | F1=0.7698 | Signal_Sig=178.5770 | Train_Time=601.89s

[Exp5-XGB] Training Version B...
[Exp5-XGB] Version B: best_threshold=0.19 | val_F1=0.3874
[Exp5-XGB Version B] AUC=0.8068 | F1=0.3983 | Signal_Sig=26.0386 | Train_Time=583.84s

[Exp5-XGB] Training Version C...
[Exp5-XGB] Version C: best_threshold=0.10 | val_F1=0.1685
[Exp5-XGB Version C] AUC=0.7690 | F1=0.1764 | Signal_Sig=4.0603 | Train_Time=458.28s

[Exp5-XGB] Complete.


In [3]:
results_df = pd.DataFrame(all_results)[['Version','AUC','F1','Signal_Significance','Train_Time_sec','Best_Threshold']]
save_path  = os.path.join(RESULTS_DIR, 'experiment5_threshold_xgb.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")

Version      AUC       F1  Signal_Significance  Train_Time_sec  Best_Threshold
      A 0.821538 0.769792           178.577019          601.89            0.36
      B 0.806795 0.398349            26.038623          583.84            0.19
      C 0.768989 0.176408             4.060293          458.28            0.10

✓ Saved → ..\results\experiment5_threshold_xgb.csv
